### 0. Environments

In [4]:
!pip install -q sentence-transformers faiss-gpu-cu12 ir_datasets

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 31.3 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.2 MB/s eta 0:00:00


In [16]:
import ir_datasets
from tqdm import tqdm
from itertools import islice
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import json
import os

In [ ]:
# configurations
SAVE_PATH: str = '/kaggle/working/artifacts'
os.makedirs(SAVE_PATH, exist_ok=True)
EMB_PATH = os.path.join(SAVE_PATH, 'embeddings.npy')
METADATA_PATH = os.path.join(SAVE_PATH, 'doc_ids.jsonl')
FAISS_INDEX_PATH = os.path.join(SAVE_PATH, 'faiss.index')

EMBEDDING_MODEL: str = 'BAAI/bge-small-en-v1.5'
BATCH_SIZE: int = 256
DEVICE = ['cuda:0', 'cuda:1']
NORMALIZE_EMB: bool = True

# config for dry test
QUICK_RUN: bool = True
MAX_SAMPLES: int = 100

### 1. Dataset

In [6]:
!cp -r /kaggle/input/datasets/hongkimgip/msmarco-passage/msmarco-passage /root/.ir_datasets/msmarco-passage

In [7]:
def preview_iter(title, iterator, fields, n=5):
    print(f"\n========== {title} ==========")

    for i, item in enumerate(islice(iterator, n), start=1):
        print(f"\n--- {title[:-1].title()} {i} ---")
        for field in fields:
            value = getattr(item, field)
            if field == "text":
                value = value[:500]
            print(f"{field}:", value)


def download_and_preview_msmarco(
    corpus_id="msmarco-passage",
    eval_id="msmarco-passage/dev/small",
    n_samples=1,
):
    passages = ir_datasets.load(corpus_id)
    eval_set = ir_datasets.load(eval_id)

    preview_iter(
        "SAMPLE PASSAGES",
        passages.docs_iter(),
        fields=["doc_id", "text"],
        n=n_samples,
    )

    preview_iter(
        "SAMPLE QUERIES",
        eval_set.queries_iter(),
        fields=["query_id", "text"],
        n=n_samples,
    )

    preview_iter(
        "SAMPLE QRELS",
        eval_set.qrels_iter(),
        fields=["query_id", "doc_id", "relevance"],
        n=n_samples,
    )

    corpus = {}
    queries = {}
    qrels = {}

    print("\n========== LOADING FULL CORPUS ==========")
    for doc in tqdm(passages.docs_iter(), desc="Loading passages"):
        corpus[doc.doc_id] = doc.text

    print(f"Total passages loaded: {len(corpus):,}")

    print("\n========== LOADING QUERIES ==========")
    for query in tqdm(eval_set.queries_iter(), desc="Loading queries"):
        queries[query.query_id] = query.text

    print(f"Total queries loaded: {len(queries):,}")

    print("\n========== LOADING QRELS ==========")
    for qrel in tqdm(eval_set.qrels_iter(), desc="Loading qrels"):
        qrels.setdefault(qrel.query_id, {})[qrel.doc_id] = qrel.relevance

    print(f"Total qrels queries loaded: {len(qrels):,}")

    return corpus, queries, qrels


corpus, _, __ = download_and_preview_msmarco()


========== SAMPLE PASSAGES ==========

--- Sample Passage 1 ---
doc_id: 0
text: The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.

========== SAMPLE QUERIES ==========

--- Sample Querie 1 ---
query_id: 1048585
text: what is paula deen's brother

========== SAMPLE QRELS ==========

--- Sample Qrel 1 ---
query_id: 300674
doc_id: 7067032
relevance: 1

========== LOADING FULL CORPUS ==========


Loading passages: 8841823it [00:34, 252667.23it/s]


Total passages loaded: 8,841,823

========== LOADING QUERIES ==========


Loading queries: 6980it [00:00, 409532.39it/s]


Total queries loaded: 6,980

========== LOADING QRELS ==========


Loading qrels: 7437it [00:00, 390493.85it/s]

Total qrels queries loaded: 6,980


### 2. Create embeddings

In [13]:
# =========================
# 1. Prepare docs
# =========================

model = SentenceTransformer(EMBEDDING_MODEL)

if QUICK_RUN:
    doc_ids = list(corpus.keys())[:MAX_SAMPLES]
else:
    doc_ids = list(corpus.keys())

texts = [corpus[doc_id] for doc_id in doc_ids]


# =========================
# 2. Encode embeddings
# =========================

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    normalize_embeddings=NORMALIZE_EMB,
    show_progress_bar=True,
)

embeddings = embeddings.astype("float32")

assert len(doc_ids) == embeddings.shape[0]


# =========================
# 3. Save embeddings + metadata
# =========================

np.save(EMB_PATH, embeddings)

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    for row_id, doc_id in enumerate(doc_ids):
        f.write(json.dumps({
            "row_id": row_id,
            "doc_id": doc_id,
        }, ensure_ascii=False) + "\n")

print("Saved embeddings:", embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chunks:   0%|          | 0/10 [00:00<?, ?it/s]

Saved embeddings: (10, 384)


#### Save as FAISS index:

In [ ]:
# =========================
# 4. Build FAISS index
# =========================

dim = embeddings.shape[1]

# Nếu embeddings đã normalize thì dùng Inner Product tương đương cosine similarity
if NORMALIZE_EMB:
    base_index = faiss.IndexFlatIP(dim)
else:
    # Nếu chưa normalize, có thể dùng L2
    base_index = faiss.IndexFlatL2(dim)

# Bọc bằng IDMap để FAISS trả về row_id đúng với metadata
index = faiss.IndexIDMap2(base_index)

row_ids = np.arange(len(doc_ids)).astype("int64")

index.add_with_ids(embeddings, row_ids)

assert index.ntotal == len(doc_ids)

faiss.write_index(index, FAISS_INDEX_PATH)

print("Saved FAISS index:", FAISS_INDEX_PATH)